Notebook 04 – Feature Engineering

- Objective

The objective of this notebook is to transform cleaned metadata and image quality measurements into a unified feature dataset for machine learning.

The engineered features generated in this notebook will serve as inputs to the baseline models and subsequently support the AI Trust Score framework.

- Input : - master_metadata.csv, - image_quality_features.csv

- Output : - feature_dataset.csv, - feature_summary.csv, - feature_correlation.csv, - feature_list.csv

In [2]:
from pathlib import Path

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = Path("../outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

In [ ]:
metadata = pd.read_csv("../data/processed/master_metadata.csv")

print(metadata.shape)

metadata.head()

(36307, 11)


,dataset,image_path,file_name,extension,target,valid_image,size_mb,resolution,width,height,aspect_ratio
0,ADMS,../data/raw/adms/Valid/images/c_-308-_jpg.rf.0...,c_-308-_jpg.rf.0b71b13d1c0db0ac91b0997247d22a4...,.jpg,Open Eye,True,0.036607,"(640, 512)",640,512,1.250000
1,ADMS,../data/raw/adms/Valid/images/112_jpg.rf.93fc7...,112_jpg.rf.93fc71b2c98ad78087aaf17c7778f857.jpg,.jpg,Open Eye,True,0.023131,"(640, 480)",640,480,1.333333
2,ADMS,../data/raw/adms/Valid/images/CHANNEL_03_20220...,CHANNEL_03_20220509152715_20220509152725_mp4-8...,.jpg,Open Eye,True,0.018838,"(640, 384)",640,384,1.666667
3,ADMS,../data/raw/adms/Valid/images/358_jpg.rf.0bc5f...,358_jpg.rf.0bc5fd26b9209862fe5336d1cacb2ed8.jpg,.jpg,Open Eye,True,0.031053,"(640, 427)",640,427,1.498829
4,ADMS,../data/raw/adms/Valid/images/_-3-_mp4-96_jpg....,_-3-_mp4-96_jpg.rf.5a65ec89c93eaf85c76b0530bda...,.jpg,NaN,True,0.020555,"(640, 523)",640,523,1.223709


In [ ]:
quality = pd.read_csv(
    "../data/processed/image_quality_features.csv"
)

print(quality.shape)

quality.head()

(36307, 4)


,image_path,brightness,contrast,blur_score
0,../data/raw/adms/Valid/images/c_-308-_jpg.rf.0...,101.796115,77.421768,303.034534
1,../data/raw/adms/Valid/images/112_jpg.rf.93fc7...,117.716758,83.798998,102.319757
2,../data/raw/adms/Valid/images/CHANNEL_03_20220...,34.701331,31.426948,2054.328330
3,../data/raw/adms/Valid/images/358_jpg.rf.0bc5f...,191.340501,45.981054,535.900534
4,../data/raw/adms/Valid/images/_-3-_mp4-96_jpg....,58.986230,60.506965,739.359342


In [ ]:
features = metadata.merge(
    quality,
    on="image_path",
    how="left"
)
print(features.shape)
features.head()

(36307, 14)


,dataset,image_path,file_name,extension,target,valid_image,size_mb,resolution,width,height,aspect_ratio,brightness,contrast,blur_score
0,ADMS,../data/raw/adms/Valid/images/c_-308-_jpg.rf.0...,c_-308-_jpg.rf.0b71b13d1c0db0ac91b0997247d22a4...,.jpg,Open Eye,True,0.036607,"(640, 512)",640,512,1.250000,101.796115,77.421768,303.034534
1,ADMS,../data/raw/adms/Valid/images/112_jpg.rf.93fc7...,112_jpg.rf.93fc71b2c98ad78087aaf17c7778f857.jpg,.jpg,Open Eye,True,0.023131,"(640, 480)",640,480,1.333333,117.716758,83.798998,102.319757
2,ADMS,../data/raw/adms/Valid/images/CHANNEL_03_20220...,CHANNEL_03_20220509152715_20220509152725_mp4-8...,.jpg,Open Eye,True,0.018838,"(640, 384)",640,384,1.666667,34.701331,31.426948,2054.328330
3,ADMS,../data/raw/adms/Valid/images/358_jpg.rf.0bc5f...,358_jpg.rf.0bc5fd26b9209862fe5336d1cacb2ed8.jpg,.jpg,Open Eye,True,0.031053,"(640, 427)",640,427,1.498829,191.340501,45.981054,535.900534
4,ADMS,../data/raw/adms/Valid/images/_-3-_mp4-96_jpg....,_-3-_mp4-96_jpg.rf.5a65ec89c93eaf85c76b0530bda...,.jpg,NaN,True,0.020555,"(640, 523)",640,523,1.223709,58.986230,60.506965,739.359342


In [ ]:
features = features[
    features["target"].isin(
        [
            "Open Eye",
            "Closed Eye"
        ]
    )
].copy()

print(features.shape)

(10889, 14)


In [ ]:
features["image_area"] = (

    features["width"] *

    features["height"]

)

In [ ]:
features["is_square"] = (

    features["aspect_ratio"].round(2) == 1.0

).astype(int)

In [ ]:
features["size_per_pixel"] = (

    features["size_mb"] /

    features["image_area"]

)

In [ ]:
features = pd.get_dummies(

    features,

    columns=["dataset"]

)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numeric = [

    "size_mb",

    "width",

    "height",

    "aspect_ratio",

    "brightness",

    "contrast",

    "blur_score",

    "image_area",

    "size_per_pixel"

]

features[numeric] = scaler.fit_transform(

    features[numeric]

)

In [ ]:
corr = features.corr(numeric_only=True)

display(corr)

corr.to_csv(

    "../outputs/tables/feature_correlation.csv"

)

,valid_image,size_mb,width,height,aspect_ratio,brightness,contrast,blur_score,image_area,is_square,size_per_pixel,dataset_ADMS,dataset_Drowsiness
valid_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
size_mb,NaN,1.000000,0.842668,0.874742,0.534546,-0.108994,0.822871,0.560750,0.874331,-0.812585,-0.760148,0.855738,-0.855738
width,NaN,0.842668,1.000000,0.878374,0.813739,-0.385120,0.758937,0.686594,0.947820,-0.964857,-0.956262,0.991219,-0.991219
height,NaN,0.874742,0.878374,1.000000,0.445871,-0.203752,0.806431,0.415984,0.976475,-0.851169,-0.877193,0.909174,-0.909174
aspect_ratio,NaN,0.534546,0.813739,0.445871,1.000000,-0.481252,0.456407,0.814238,0.590898,-0.791715,-0.739508,0.771319,-0.771319
brightness,NaN,-0.108994,-0.385120,-0.203752,-0.481252,1.000000,-0.004322,-0.508161,-0.284684,0.406646,0.410471,-0.380176,0.380176
contrast,NaN,0.822871,0.758937,0.806431,0.456407,-0.004322,1.000000,0.359952,0.798993,-0.730095,-0.655302,0.771460,-0.771460
blur_score,NaN,0.560750,0.686594,0.415984,0.814238,-0.508161,0.359952,1.000000,0.509845,-0.683040,-0.606658,0.668072,-0.668072
image_area,NaN,0.874331,0.947820,0.976475,0.590898,-0.284684,0.798993,0.509845,1.000000,-0.905470,-0.934963,0.960289,-0.960289
is_square,NaN,-0.812585,-0.964857,-0.851169,-0.791715,0.406646,-0.730095,-0.683040,-0.905470,1.000000,0.942002,-0.974236,0.974236


In [ ]:
features.to_csv(

    "../data/processed/feature_dataset.csv",

    index=False

)

print("✓ feature_dataset.csv created")

NameError: name 'features' is not defined